In [1]:
# Training the Jawn Do wake word (microWakeWord) -- the ready-to-run Colab copy.
# Same pipeline as Hey Jawn, with two changes:
#   target_word = 'jawn do'   and   dst = 'jawn_do.tflite'
# Run all cells on a GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# The last cell hands jawn_do.tflite to your browser. Check it is NOT the same
# bytes as hey_jawn.tflite (every tflite in ~/Downloads was md5 a3da6184... -
# that copy trap is exactly how the old Jawn Do came to be Hey Jawn).

# 1. Install microWakeWord and its pinned dependencies
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!pip install -q pymicro-features mmap-ninja datasets soundfile
!test -d microWakeWord || git clone -q https://github.com/kahrendt/microWakeWord
!pip install -q -e ./microWakeWord

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# 2. Piper sample generator (the positive samples) + our target word
import os, sys

target_word = 'jawn do'   # phonetic spellings often sample better; try variants if training struggles

if not os.path.exists("./piper-sample-generator"):
    !git clone -q https://github.com/rhasspy/piper-sample-generator
    !mkdir -p piper-sample-generator/models
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !pip install -q torch torchaudio piper-phonemize-cross==1.2.1
    sys.path.append("piper-sample-generator/")

ERROR: Could not find a version that satisfies the requirement piper-phonemize-cross==1.2.1 (from versions: none)
ERROR: No matching distribution found for piper-phonemize-cross==1.2.1


In [3]:
# 3. Generate one sample and listen to it (checks the phrase sounds right)
!python3 piper-sample-generator/generate_samples.py "{target_word}" --max-samples 1 --batch-size 1 --output-dir generated_samples
from IPython.display import Audio
Audio("generated_samples/0.wav", autoplay=True)

python3: can't open file '/content/piper-sample-generator/generate_samples.py': [Errno 2] No such file or directory


ValueError: rate must be specified when data is a numpy array or list of audio samples.

In [ ]:
# 4. Generate many samples. More samples (and different pronunciations) help.
# 3000 is a reasonable first pass; the upstream default of 1000 is often too few.
!python3 piper-sample-generator/generate_samples.py "{target_word}" --max-samples 3000 --batch-size 100 --output-dir generated_samples

In [ ]:
# 5. Pre-generated negative spectrograms (speech, party noise, no-speech)
import os
output_dir = './negative_datasets'
os.makedirs(output_dir, exist_ok=True)
link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
for fname in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
    !wget -q -O "{output_dir}/{fname}" "{link_root}{fname}"
    !unzip -q -o "{output_dir}/{fname}" -d "{output_dir}"
print("negatives:", os.listdir(output_dir))

In [ ]:
# 6. Impulse responses and background noise for augmentation (this is slow)
import os
import datasets, scipy
import numpy as np
from pathlib import Path
from tqdm import tqdm

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

if not os.path.exists("audioset"):
    os.mkdir("audioset")
    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    !wget -q -O {out_dir} "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}"
    !cd audioset && tar -xf {fname}
    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    !rm -rf audioset

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    !wget -q -O {output_dir}/{fname} "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/{fname}"
    !cd {output_dir} && unzip -q {fname}
    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# 7. Clips + augmentation
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1)

augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities={
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                         },
                         impulse_paths=['mit_rirs'],
                         background_paths=['fma_16k', 'audioset_16k'],
                         background_min_snr_db=-5,
                         background_max_snr_db=10,
                         min_jitter_s=0.195,
                         max_jitter_s=0.205)

In [ ]:
# 8. Augment samples and write the training/validation/testing feature sets
import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'
os.makedirs(output_dir, exist_ok=True)
for split in ["training", "validation", "testing"]:
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)
    split_name = "train"
    repetition = 2
    spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10)
    if split == "validation":
        split_name = "validation"
        repetition = 1
    elif split == "testing":
        split_name = "test"
        repetition = 1
        spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=1, step_ms=10)
    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True)

In [ ]:
# 9. Write the training config (hyperparameters to experiment with)
import yaml, os

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/wakeword"

config["features"] = [
    {"features_dir": "generated_augmented_features", "sampling_weight": 2.0, "penalty_weight": 1.0, "truth": True,  "truncation_strategy": "truncate_start", "type": "mmap"},
    {"features_dir": "negative_datasets/speech",          "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
    {"features_dir": "negative_datasets/dinner_party",    "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
    {"features_dir": "negative_datasets/no_speech",        "sampling_weight": 5.0,  "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
    {"features_dir": "negative_datasets/dinner_party_eval","sampling_weight": 0.0,  "penalty_weight": 1.0, "truth": False, "truncation_strategy": "split",  "type": "mmap"},
]
config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 128
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)
print("wrote training_parameters.yaml")

In [ ]:
# 10. Train (~10k steps), then quantise and convert to a streaming TFLite model
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# 11. Your model: copied to the working directory and offered for download.
#     The ROC file the training prints picks the probability_cutoff in
#     jw/voice_models.h (0.55 today): raise it if it false-triggers, lower it
#     if it misses.
import os, shutil
src = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
dst = "jawn_do.tflite"
if os.path.exists(src):
    shutil.copy(src, dst)
    print("model ready:", os.path.abspath(dst), os.path.getsize(dst), "bytes")
    print("also check trained_models/wakeword/tflite_stream_state_internal_quant/tflite_streaming_roc.txt")
    try:
        from google.colab import files
        files.download(dst)
    except Exception:
        print("not running on Colab: download", os.path.abspath(dst), "from the file browser")
else:
    print("not found - did the training cell finish?")